# 🎙️ VoiceBridge: Dysarthric Speech LoRA Fine-Tuning Pipeline
### THREWS Odyssey 2026 — Track: Human-AI & Assistive Technology
**Team:** Sleeper Cells (SIES Graduate School of Technology)
* **Talha Shaikh:** Lead System Architect
* **Aaliya Hussain:** Machine Learning & Model Adaptation Lead
* **Moulika Sanyal:** Edge AI Optimization & ONNX Quantization Lead
* **Manas Suryawanshi:** Clinical Research & Pitch Lead

---

### Notebook Objectives
1. **Baseline Proof:** Stream University of Toronto's **TORGO** dysarthric dataset and prove standard Whisper achieves >70% Word Error Rate (WER).
2. **Parameter-Efficient Fine-Tuning (PEFT / LoRA):** Freeze ~99.28% of `openai/whisper-small` (or `openai/whisper-tiny` for constrained VRAM) and train lightweight adapter matrices ($r=16, \alpha=32$) on acoustic attention projections (`q_proj`, `v_proj`) across encoder and decoder.
3. **Export Checkpoint:** Save lightweight ~7.1MB adapter weights (safetensors) for direct backend integration, enabling low-latency CPU/edge execution without discrete GPUs.

> **Colab T4 GPU Required:** Go to `Runtime` -> `Change runtime type` -> select **T4 GPU**.

## 1. Environment Setup & Dependency Installation

In [ ]:
# Install state-of-the-art Hugging Face audio and PEFT tooling
!pip install -q datasets transformers evaluate soundfile jiwer torchaudio accelerate peft optimum onnxruntime

import torch
print(f"[+] PyTorch Version: {torch.__version__}")
print(f"[+] CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[+] Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[!] Warning: GPU runtime not active. Please switch to T4 GPU in Colab settings.")

## 2. Stream TORGO Dysarthric Dataset from Hugging Face Hub

In [ ]:
from datasets import load_dataset

print("[+] Connecting to Hugging Face Hub ('abnerh/TORGO-database')...")
dataset = load_dataset("abnerh/TORGO-database", split="train", streaming=True)
print("[+] Connected successfully! Streaming audio without needing 50GB local download.")

## 3. Listen to Dysarthric Speech Samples (Acoustic Distortion Proof)

In [ ]:
import IPython.display as ipd

samples = list(dataset.take(3))
for i, sample in enumerate(samples):
    audio = sample["audio"]
    text = sample.get("text", sample.get("transcription", "Unknown"))
    print(f"\n--- Sample {i+1} ---")
    print(f"Target Sentence: \"{text}\"")
    print(f"Sample Rate: {audio['sampling_rate']} Hz | Duration: {len(audio['array']) / audio['sampling_rate']:.2f}s")
    display(ipd.Audio(audio["array"], rate=audio["sampling_rate"]))

## 4. Empirical Baseline Proof: The 70%+ Foundation Model Failure Wall

In [ ]:
from transformers import pipeline
import evaluate

print("[+] Loading pre-trained foundation Whisper-tiny...")
device_id = 0 if torch.cuda.is_available() else -1
asr_baseline = pipeline("automatic-speech-recognition", model="openai/whisper-tiny", device=device_id)
wer_metric = evaluate.load("wer")

refs, preds = [], []
print("\n--- Running Baseline Zero-Shot Evaluation ---")
for i, sample in enumerate(samples):
    target = sample.get("text", sample.get("transcription", ""))
    pred = asr_baseline({"array": sample["audio"]["array"], "sampling_rate": sample["audio"]["sampling_rate"]})["text"]
    refs.append(target)
    preds.append(pred)
    print(f"[Sample {i+1}]")
    print(f"  Target:   '{target}'")
    print(f"  Baseline: '{pred}'")

baseline_wer = wer_metric.compute(predictions=preds, references=refs)
print("=" * 60)
print(f"[!] Baseline Whisper Word Error Rate (WER): {baseline_wer * 100:.1f}%")
print("    (Empirical proof that standard models fail on non-normative speech)")
print("=" * 60)

## 5. Feature Extractor & Audio Preprocessing Pipeline

In [ ]:
from transformers import WhisperProcessor

print("[+] Initializing WhisperProcessor (16kHz Log-Mel Feature Extractor & Tokenizer)...")
BASE_MODEL = "openai/whisper-small"  # or "openai/whisper-tiny"
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="english", task="transcribe")

def prepare_sample(item):
    audio = item["audio"]
    # Extract 80-channel log-mel spectrogram
    input_features = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    # Tokenize target text
    text = item.get("text", item.get("transcription", ""))
    labels = processor.tokenizer(text).input_ids
    return {"input_features": input_features, "labels": labels}

print("[+] Preparing dataset cache (400 samples for high-impact Colab adaptation)...")
raw_items = list(dataset.take(400))
processed_items = [prepare_sample(x) for x in raw_items]

split = int(0.85 * len(processed_items))
train_dataset = processed_items[:split]
eval_dataset = processed_items[split:]
print(f"[+] Cached: {len(train_dataset)} Train Samples | {len(eval_dataset)} Validation Samples")

## 6. Dynamic Sequence Padding Collator

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

## 7. Model Initialization & PEFT / LoRA Injection

In [ ]:
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

# Configurable model tier: "openai/whisper-small" (clinical production) or "openai/whisper-tiny" (fast Colab demo)
MODEL_ID = "openai/whisper-small"
print(f"[+] Loading base Whisper architecture ({MODEL_ID})...")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# Configure Low-Rank Adaptation (LoRA) targeting encoder query/value projections
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

## 8. Training Execution Loop (Seq2SeqTrainer)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

training_args = Seq2SeqTrainingArguments(
    output_dir="./voicebridge_training_checkpoints",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-3,
    warmup_steps=30,
    max_steps=150,
    fp16=torch.cuda.is_available(),
    evaluation_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=2,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to=["none"],
    load_best_model_at_end=True
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=peft_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

print("[*] Initiating LoRA Training Loop on Colab GPU...")
trainer.train()

## 9. Export Adapter Weights & Package for Edge Quantization

In [ ]:
import os
import shutil

adapter_export_dir = "./voicebridge_lora_weights"
peft_model.save_pretrained(adapter_export_dir)
processor.save_pretrained(adapter_export_dir)
print(f"[+] LoRA Adapter successfully exported to: {adapter_export_dir}")

# Create zip archive for immediate download from Colab file manager
shutil.make_archive("voicebridge_lora_weights", "zip", adapter_export_dir)
print("[+] Downloadable archive ready: voicebridge_lora_weights.zip")
print("[!] Next Step: Share this zip with Moulika for ONNX INT8 quantization!")